# Demo — end-to-end music-context inference

One audio file + its textual context → music-structure graph → GNN–BERT fusion →
predicted genre/mood tags (and valence/arousal where the model has an emotion head).

**Prerequisites**

```powershell
python -m src.prepare_data --stage all --jobs 10
python -m src.train_task3 --dataset deam --fusion cross_attn --save_model
```

The checkpoint is read from `results/checkpoints/task3_<dataset>_cross_attn.pt`.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, torch
from src.config import CFG, path
from src.utils import get_device, quiet_transformers

quiet_transformers()
DEVICE = get_device()
DATASET = 'deam'          # dataset the checkpoint was trained on
print('device:', DEVICE)

## 1. Audio → segments → chords → graph

In [ ]:
from src.audio_features import extract_track, chord_sequence_names
from src.graph_builder import build_track_graph, to_pyg

AUDIO = path('deam_audio') / '10.mp3'        # <- any wav/mp3 on disk
TEXT  = 'Track: Wish. Artist: Nine Inch Nails. Listener tags: industrial, angry, loud.'

feat = extract_track(AUDIO)
rec = build_track_graph(feat['x'], feat['chords'])
rec['id'] = AUDIO.stem
data = to_pyg(rec)

print(f"segments (nodes): {data.num_nodes}")
print(f"edges           : {data.edge_index.shape[1]}")
print(f"node feature dim: {data.x.shape[1]}")
print('chord path      :', ' -> '.join(chord_sequence_names(feat['chords'])[:12]))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].imshow(feat['mel'], aspect='auto', origin='lower', cmap='magma')
ax[0].set_title('log-mel spectrogram'); ax[0].set_xlabel('frame'); ax[0].set_ylabel('mel bin')

A = np.zeros((data.num_nodes, data.num_nodes))
ei, ea = data.edge_index.numpy(), data.edge_attr.numpy()
A[ei[0], ei[1]] = ea.max(1)
im = ax[1].imshow(A, cmap='Blues'); ax[1].set_title('segment graph adjacency (max edge weight)')
ax[1].set_xlabel('segment'); ax[1].set_ylabel('segment'); fig.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

## 2. Load the trained fusion model

In [ ]:
from src.bert_encoder import make_tokenizer
from src.fusion_model import FusionModel
from src.graph_dataset import build_labels, fit_feature_stats, apply_feature_stats
from src.audio_features import CONT_DIM
from src.graph_builder import load_graphs

_, spec = build_labels(DATASET, label_mode='multi')
CLASSES = spec.classes

# reuse the training-split feature statistics so the demo matches training
graphs = load_graphs(DATASET)
mu, sd, cont_dim = fit_feature_stats(list(graphs.values())[:1000], CONT_DIM)
data.x = apply_feature_stats(data.x, mu, sd, cont_dim)

ckpt = ROOT / 'results' / 'checkpoints' / f'task3_{DATASET}_cross_attn.pt'
model = FusionModel(in_dim=data.x.shape[1], n_tags=len(CLASSES), mode='cross_attn',
                    hidden=CFG['task3']['hidden'], gnn_layers=CFG['task3']['gnn_layers'],
                    n_va=2 if spec.va is not None else 0).to(DEVICE)
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print('loaded', ckpt.name)
else:
    print('WARNING: no checkpoint — run train_task3 with --save_model first')
model.eval();

## 3. Inference: tags, emotion, and what the graph attends to

In [ ]:
from torch_geometric.data import Batch

tok = make_tokenizer(CFG['text']['model_name'])
enc = tok(TEXT, truncation=True, padding='max_length',
          max_length=CFG['text']['max_length'], return_tensors='pt')
data.input_ids = enc['input_ids']
data.attention_mask = enc['attention_mask']
batch = Batch.from_data_list([data]).to(DEVICE)

with torch.no_grad():
    out = model(batch, output_attentions=True)
probs = torch.sigmoid(out['logits'])[0].cpu().numpy()

print('TOP PREDICTED CONTEXT TAGS')
for i in np.argsort(-probs)[:8]:
    print(f'  {CLASSES[i]:<18} {probs[i]:.3f}')

if out['va'] is not None:
    v, a = out['va'][0].cpu().numpy()
    print(f'\nvalence {v:+.3f}  arousal {a:+.3f}   (scale [-1, 1];'
          f' 1-9 equivalent: {v*4+5:.2f} / {a*4+5:.2f})')

In [ ]:
attn = out['text_attention'][0].cpu().numpy()
mask = enc['attention_mask'][0].bool()
toks = tok.convert_ids_to_tokens(enc['input_ids'][0][mask])
w = attn[:int(mask.sum())]
print('TEXT TOKENS THE GRAPH READOUT ATTENDS TO')
for i in np.argsort(-w)[:10]:
    if toks[i] not in ('[CLS]', '[SEP]', '[PAD]'):
        print(f'  {toks[i]:<16} {w[i]:.4f}')

## 4. Where the numbers live

* `results/metrics/*.json` — one file per experiment (args, history, test metrics, baselines)
* `results/comparison/*.md` — cross-dataset tables for tasks 1–3
* `results/plots/` — F1 curves, confusion matrices, t-SNE, bar charts
* `results/case_studies/` — chord paths + text alignment for individual tracks